## Notebook conventions

**Edit this file in place — don't save a new copy** (no `_V1`/`_V2`/dated/`-Copy`/`DEBUG-` variants). Commit changes via a branch + PR; git history is the version record, not the filename. Full conventions and `nbstripout` setup: see the repo [README](../../README.md) → "Notebook conventions."

# Batch Image Processing with MegaDetector

This notebook runs MegaDetector on a folder of camera trap images, then renders annotated output images (for visual review / Timelapse-style browsing) rather than just producing the raw detections JSON.

**If you are unsure what any of this does, see First_time_setup first.**

This version calls the **pip-installed** `megadetector` package (via `python -m ...`) using the exact Python interpreter behind the current kernel (`sys.executable`), instead of invoking script files from an old GitHub clone. This avoids two issues seen previously on Tarazed:

1. A stray machine-wide `PYTHONPATH` (now removed) that used to make old-clone-style invocations work by accident, while silently shadowing pip installs elsewhere.
2. Bare `!python ...` shell calls resolving to whichever Python happens to be first on the shell's PATH, which is not necessarily the environment behind the notebook's selected kernel.

Input/output folders are chosen interactively via file-picker dialogs (same pattern as `Video_Processing_Windows.ipynb`), rather than hardcoded paths - this avoids stale/missing-folder errors from paths left over from a previous dataset.

## Setup

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import tkinter as tk
from tkinter import filedialog as fd

# Create a hidden root window for file dialogs
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

print("Imports successful.")

In [ ]:
# Sanity check: confirm this kernel is backed by the environment that has
# megadetector pip-installed (should point into the "megadetector" env).
print(sys.executable)

In [ ]:
# Confirm megadetector is actually installed in this environment.
# Note: megadetector.__version__ is not reliable/does not exist on the module -
# use importlib.metadata instead.
import importlib.metadata
print("megadetector version:", importlib.metadata.version("megadetector"))

## Select folders

A dialog will pop up (it may appear behind other windows - check your taskbar) for each selection below.

In [ ]:
# Select input folder containing images to process
input_folder = fd.askdirectory(title="Select folder containing image files")
if not input_folder:
    raise ValueError("No input folder selected")

folder_path = os.path.abspath(input_folder)
print("Input folder:", folder_path)

In [ ]:
# Select output folder - this is where the detections JSON and rendered/annotated
# images will be written
output_folder = fd.askdirectory(title="Select folder for output (JSON + rendered images)")
if not output_folder:
    raise ValueError("No output folder selected")

OUTPUT_DIR = Path(output_folder)
OUTPUT_DIR.mkdir(exist_ok=True)

run_name = Path(folder_path).name
output_json = str(OUTPUT_DIR / f"{run_name}.json")
output_path = str(OUTPUT_DIR / "detections")

print("Output folder:", OUTPUT_DIR)
print("Detections JSON will be written to:", output_json)
print("Rendered images will be written to:", output_path)

In [ ]:
# Select the MegaDetector model weights file (.pt).
# Using a local file avoids the auto-download SSL cert error seen previously
# on this machine when using a keyword like MDV5A.
model_file = fd.askopenfilename(
    title="Select MegaDetector model file (.pt)",
    filetypes=[("PyTorch model", "*.pt"), ("All files", "*.*")],
    initialdir=r"C:\Users\Public\Documents\MegaDetector_App\Models"
)
if not model_file:
    raise ValueError("No model file selected")

model_path = os.path.abspath(model_file)
print("Model file:", model_path)

In [ ]:
# Verify everything actually exists before running anything.
print("folder_path:", folder_path, "| exists:", os.path.isdir(folder_path))
print("model_path:", model_path, "| exists:", os.path.isfile(model_path))

assert os.path.isdir(folder_path), f"Input folder not found: {folder_path}"
assert os.path.isfile(model_path), f"Model file not found: {model_path}"

## Run detection

Runs MegaDetector over every image in `folder_path` and writes the raw results to `output_json`. `--checkpoint_frequency 1000` writes periodic checkpoints in case of a crash on a large batch.

In [ ]:
result = subprocess.run([
    sys.executable, "-m", "megadetector.detection.run_detector_batch",
    model_path,
    folder_path,
    output_json,
    "--output_relative_filenames",
    "--recursive",
    "--checkpoint_frequency", "1000",
    "--quiet",
], capture_output=True, text=True)

print("RETURN CODE:", result.returncode)
print("---STDOUT---")
print(result.stdout)
print("---STDERR---")
print(result.stderr)

assert result.returncode == 0, "Detection failed - see stderr above."

## Postprocess / render annotated images

Reads `output_json` and renders annotated copies of the images (with bounding boxes) plus an `index.html` viewer into `output_path`. Only run this after confirming the detection cell above finished with return code 0.

In [ ]:
result = subprocess.run([
    sys.executable, "-m", "megadetector.postprocessing.postprocess_batch_results",
    output_json,
    output_path,
    "--image_base_dir", folder_path,
    "--num_images_to_sample", "-1",
], capture_output=True, text=True)

print("RETURN CODE:", result.returncode)
print("---STDOUT---")
print(result.stdout)
print("---STDERR---")
print(result.stderr)

assert result.returncode == 0, "Postprocessing failed - see stderr above."

## Done

If both cells above returned code 0, open `output_path/index.html` in a browser to review the annotated images.

In [ ]:
print("Review results at:", os.path.join(output_path, "index.html"))